# FLAG 2027 - Notebook 42 RUN-ALL: public face-voice models as frozen blocks (research round 10)

Every encoder we use was trained on ONE modality. Three public models were trained on VoxCeleb to put faces and voices in one space, which is exactly what our head has to learn from 70 challenge identities plus external rows. None of the FAME or FLAG papers has used them as frozen feature blocks. All three were trained without any Bengali speech, so they are allowed under the FAQ (items 3, 6, 7) and will be declared.

| Block | Model | Trained on | Output |
|---|---|---|---|
| `vv_face`, `vv_voice` | VisualVoice cross-modal pretraining (Gao & Grauman, CVPR 2021) | VoxCeleb2, face-voice matching loss | 128-d each |
| `cmpc_face`, `cmpc_voice` | CMPC (Zhu et al., IJCAI 2022) | VoxCeleb1 talking faces, unsupervised | 512-d each, L2 |
| `sss_face`, `sss_voice` | Seeking the Shape of Sound (Wen et al., CVPR 2021) | VoxCeleb1 + VGGFace2, identity-supervised | 128-d each, L2 |

What the notebook does:

1. Rebuilds the face and voice file lists for the challenge train, MAV v1/v3 and MAV extra rows (NB39 Cell 2, with the CLIP and Whisper alignment checks).
2. Loads the three models from their public checkpoints (architectures and preprocessing copied from the repositories, checkpoints loaded strictly) and embeds every row with each block. Resumable per block and split; features go to `features/xmodal.npz`.
3. Zero-training check: each model's own face-voice cosine is scored directly on random and same-gender pairs of challenge rows. A block that cannot beat chance here on its own cannot help the head much either.
4. The usual CV (stage A, then stage B on new folds and seeds) in system E's recipe: each block added to E's inputs on the voice side, the face side, or both. Same adoption rule as always. A winner needs re-extraction on the VoxCeleb parts (faces must be re-pulled) before it can enter NB20.

**How to run:** T4 GPU, Runtime > Run all, about 3 hours (downloads 1.5 GB, extraction about 1.5 h, CV about 1.5 h). If the Shape-of-Sound checkpoint cannot be fetched from Google Drive, the notebook says so and continues with the other two. Paste the **SUMMARY**.


In [ ]:
# Cell 1 [NB42 v1 6-Nov] - Drive, GPU (hard check). No token needed: all three checkpoints are public files.
from google.colab import drive
drive.mount('/content/drive')
import torch, shutil, torchvision, torchaudio
assert torch.cuda.is_available(), 'NO GPU in this runtime. Runtime > Change runtime type > T4 GPU > Save, then run again.'
dev = 'cuda'
print('[NB42 v1 6-Nov] Device:', torch.cuda.get_device_name(0), '| torch', torch.__version__, '| torchvision', torchvision.__version__, '| torchaudio', torchaudio.__version__, '| disk free %.0f GB' % (shutil.disk_usage('/content')[2] / 1e9))


In [ ]:
# Cell 2 [NB42 v1 6-Nov] - face AND voice file lists in our row order (challenge train, MAV v1/v3, MAV extra; dev rows are not needed for CV) - NB39 Cell 2 verbatim
#          + alignment checks: CLIP on 100 faces per split and Whisper layers 20-28 on 48 voices per split must reproduce the stored features
import os, re, json, time, zipfile
import numpy as np, pandas as pd, cv2, librosa
from PIL import Image
from tqdm.auto import tqdm
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027'); FEATDIR = os.path.join(PROJECT, 'features'); RESDIR = os.path.join(PROJECT, 'results'); CACHE = os.path.join(PROJECT, 'models_cache')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
MAV_ZIPS, MAV_LISTS = os.path.join(BASE, 'dataset'), os.path.join(BASE, 'face_voice_association_splits')
PARTS = os.path.join(FEATDIR, 'nb42_parts'); os.makedirs(PARTS, exist_ok=True)
EXTRACT_NOTES = []
def enote(msg):
    print(msg); EXTRACT_NOTES.append(msg)
# ---- challenge train: faces and voices ----
FULL = '/content/flag_data'; os.makedirs(FULL, exist_ok=True)
if not os.path.exists(os.path.join(FULL, 'train_set.zip.done')):
    with zipfile.ZipFile(os.path.join(ZIP_DIR, 'train_set.zip')) as zf:
        zf.extractall(FULL, members=[m for m in zf.namelist() if m.lower().endswith(('.jpg', '.jpeg', '.png', '.wav', '.txt', '.csv'))])
    open(os.path.join(FULL, 'train_set.zip.done'), 'w').close()
TXT = [os.path.join(r, f) for r, _, fs in os.walk(FULL) for f in fs if f.endswith(('.txt', '.csv'))]
def find_one(*parts):
    hits = [p for p in TXT if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def resolve(rel, list_path):
    q = os.path.dirname(list_path)
    while q.startswith(FULL):
        c = os.path.join(q, rel)
        if os.path.exists(c):
            return c
        q = os.path.dirname(q)
def tok(tokens, exts, fallback=3):
    t = [x for x in tokens if x.lower().endswith(exts)]; return t[0] if len(t) == 1 else tokens[fallback]   # NB35 used token 3 for the face; NB24 the single .wav token
IMG, WAV = ('.jpg', '.jpeg', '.png'), ('.wav',)
SPLITS = ['train', 'mav', 'extra']
lp = find_one('train_English.txt'); lines = [l.split() for l in open(lp) if l.strip()]
FACES = {'train': [resolve(tok(t, IMG), lp) for t in lines]}; WAVS = {'train': [resolve(tok(t, WAV), lp) for t in lines]}
# ---- MAV v1/v3 (same rows as mavceleb_v1v3.npz) ----
def zip_index(zf):
    m = {}
    for n in zf.namelist():
        for key in ('voices/', 'faces/'):
            i = n.find(key)
            if i >= 0 and not n.endswith('/'):
                m[n[i:]] = n; break
    return m
MAVD = '/content/mav'; mav_f, mav_v, mspk = [], [], []
for ver, zname, lists in [('v1', 'mavceleb_v1_train.zip', ['English_train.txt', 'Urdu_train.txt']), ('v3', 'mavceleb_v3_tran.zip', ['English_train.txt', 'German_train.txt'])]:
    flag = os.path.join(MAVD, ver + '.both.done'); need = not os.path.exists(flag)
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); zi = zip_index(zf)
    for lf in lists:
        for l in open(os.path.join(MAV_LISTS, ver, lf)):
            t = l.split()
            if len(t) < 2:
                continue
            vm, fm = zi.get('voices/' + t[0]), zi.get('faces/' + t[1])
            if vm is None or fm is None:
                continue
            if need:
                zf.extract(vm, os.path.join(MAVD, ver)); zf.extract(fm, os.path.join(MAVD, ver))
            mav_v.append(os.path.join(MAVD, ver, vm)); mav_f.append(os.path.join(MAVD, ver, fm)); mspk.append(ver + '_' + t[0].split('/')[0])
    zf.close()
    if need:
        open(flag, 'w').close()
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz')))
assert list(MAV['spk']) == mspk, 'MAV v1/v3 row order differs - tell Claude'
FACES['mav'], WAVS['mav'] = mav_f, mav_v
# ---- MAV v2 + v1-complete 'extra' rows: the SAME random face per clip as the original extraction (same rng sequence as NB24/NB29/NB35) ----
EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz')))
MAX_PER_ID = 120
VPAT = re.compile(r'voices/(id\d+)/([^/]+)/([^/]+)/([^/]+\.wav)$'); FPAT = re.compile(r'faces/(id\d+)/([^/]+)/([^/]+)/([^/]+\.jpg)$')
v1_used = set()
for lf in ['English_train.txt', 'Urdu_train.txt']:
    for l in open(os.path.join(MAV_LISTS, 'v1', lf)):
        t = l.split()
        if t:
            v1_used.add(t[0].split('/')[0])
rows = []
for ver, zname, skip_ids in [('v2', 'mavceleb_v2_complete.zip', set()), ('v1', 'mavceleb_v1_complete.zip', v1_used)]:
    D = '/content/mavx/' + ver; flag = os.path.join(D, 'both.done'); need = not os.path.exists(flag)
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); voices, faces = {}, {}
    for n in zf.namelist():
        mv = VPAT.search(n)
        if mv:
            voices.setdefault(mv.group(1), []).append((mv.group(2), mv.group(3), n)); continue
        mf = FPAT.search(n)
        if mf:
            faces.setdefault((mf.group(1), mf.group(2), mf.group(3)), []).append(n)
    ids_ = sorted(i for i in voices if i not in skip_ids); rng = np.random.default_rng(0)
    for pid in ids_:
        vs = voices[pid]; vs = [vs[j] for j in rng.permutation(len(vs))[:MAX_PER_ID]]
        for lang_, vid, vn in vs:
            fr = faces.get((pid, lang_, vid)) or [f for (p, l, v), fl in faces.items() if p == pid for f in fl]
            if not fr:
                continue
            fn = fr[rng.integers(len(fr))]
            if need:
                zf.extract(vn, D); zf.extract(fn, D)
            rows.append((ver + '_' + pid, os.path.join(D, vn), os.path.join(D, fn)))
    zf.close()
    if need:
        open(flag, 'w').close()
spk_all = np.array([r[0] for r in rows]); keep = np.ones(len(rows), bool)
if len(rows) != len(EXTRA['spk']):
    j = 0; keep[:] = False
    for i in range(len(rows)):
        if j < len(EXTRA['spk']) and spk_all[i] == EXTRA['spk'][j]:
            keep[i] = True; j += 1
    assert j == len(EXTRA['spk']), 'could not align MAV extra rows - tell Claude'
WAVS['extra'] = [r[1] for r, k in zip(rows, keep) if k]; FACES['extra'] = [r[2] for r, k in zip(rows, keep) if k]
assert [r[0] for r, k in zip(rows, keep) if k] == list(EXTRA['spk']), 'MAV extra row order differs - tell Claude'
for s in SPLITS:
    assert None not in FACES[s] and None not in WAVS[s], 'missing face/voice files in %s - tell Claude' % s
    assert len(FACES[s]) == len(WAVS[s])
enote('rows: ' + ', '.join('%s %d' % (s, len(FACES[s])) for s in SPLITS))
def cos_rows(a, b):
    return (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-9)
# ---- alignment check 1: CLIP on 100 random faces per split must reproduce the stored CLIP rows ----
from transformers import CLIPModel, CLIPProcessor
CLIPM = CLIPModel.from_pretrained('openai/clip-vit-large-patch14').half().to(dev).eval(); CLIPP = CLIPProcessor.from_pretrained('openai/clip-vit-large-patch14')
@torch.no_grad()
def clip_embed(pils):
    x = CLIPP(images=pils, return_tensors='pt')['pixel_values'].to(dev, torch.float16)
    return CLIPM.visual_projection(CLIPM.vision_model(pixel_values=x).pooler_output).float().cpu().numpy()
CLIP_SAVED = np.load(os.path.join(FEATDIR, 'clip.npz'))
stored_f = {'train': CLIP_SAVED['train'], 'mav': MAV['clip'], 'extra': EXTRA['clip']}
rng_c = np.random.default_rng(1)
for s in SPLITS:
    saved = stored_f[s]; assert len(saved) == len(FACES[s]), '%s: %d stored CLIP rows vs %d face files - tell Claude' % (s, len(saved), len(FACES[s]))
    idx = rng_c.choice(len(saved), 100, replace=False)
    med = np.median(cos_rows(clip_embed([Image.open(FACES[s][i]).convert('RGB') for i in idx]), saved[idx]))
    enote('alignment faces  %-6s median CLIP cosine %.4f' % (s, med)); assert med > 0.98, 'face files do not match the stored CLIP rows for %s - tell Claude' % s
# ---- alignment check 2: Whisper layers 20-28 on 48 random voices per split must reproduce the stored whisper rows (same code as NB24) ----
def get_whisper():
    from transformers import WhisperFeatureExtractor, WhisperModel
    name = 'openai/whisper-large-v3'
    return WhisperFeatureExtractor.from_pretrained(name), WhisperModel.from_pretrained(name, torch_dtype=torch.float16).encoder.to(dev).eval()
def whisper_batch(fe, enc, paths, bands):
    """one batch of clips -> one pooled vector per band of layers (mean of layers lo..hi-1, mean over the clip's real frames)"""
    wavs = [librosa.load(p, sr=16000, mono=True)[0][:16000 * 30] for p in paths]
    x = fe(wavs, sampling_rate=16000, return_tensors='pt').input_features.to(dev, torch.float16)
    with torch.no_grad():
        hs = enc(x, output_hidden_states=True).hidden_states
    outs = []
    for lo, hi in bands:
        h = torch.stack(hs[lo:hi]).float().mean(0); o = []
        for k_, w in enumerate(wavs):
            n = max(1, min(1500, len(w) // 320)); o.append(h[k_, :n].mean(0).cpu().numpy())
        outs.append(np.stack(o))
    return outs
WFE, WENC = get_whisper()
WH_SAVED = np.load(os.path.join(FEATDIR, 'whisper.npz'))
rng_w = np.random.default_rng(2)
for s in SPLITS:
    saved = WH_SAVED[s]; assert len(saved) == len(WAVS[s]), '%s: %d stored whisper rows vs %d voice files - tell Claude' % (s, len(saved), len(WAVS[s]))
    idx = rng_w.choice(len(saved), 48, replace=False)
    new = np.concatenate([whisper_batch(WFE, WENC, [WAVS[s][i] for i in idx[b:b + 8]], [(20, 29)])[0] for b in range(0, 48, 8)])
    med = np.median(cos_rows(new, saved[idx])); enote('alignment voices %-6s median Whisper cosine %.4f' % (s, med)); assert med > 0.98, 'voice files do not match the stored whisper rows for %s - tell Claude' % s


In [ ]:
# Cell 3 [NB42 v1 6-Nov] - the three public face-voice models (architectures and preprocessing copied from their repositories, checkpoints loaded strictly)
#          followed by the extraction of every block on every row and a zero-training cross-modal check. Resumable per block and split.
"""
nb42_loaders.py  --  frozen face / voice embeddings from three public face-voice models.

    A. VisualVoice cross-modal pretraining      (facebookresearch/VisualVoice, main)   -> 128-d
    B. CMPC, IJCAI 2022                         (Cocoxili/CMPC, main, release v1.0.0)  -> 512-d
    C. Seeking the Shape of Sound, CVPR 2021    (KID-7391/seeking-the-shape-of-sound)  -> 128-d

Every architectural / preprocessing choice below is copied from the repository source
and annotated with  "<repo> <file> L<line>"  (line numbers as of 2026-11-06).
Checkpoint key names were verified by parsing the actual checkpoint pickles:
  facial.pth / vocal.pth           : legacy torch format, plain OrderedDict state_dict
  checkpoint_CMPC.pth.tar          : zip format, dict {epoch, model, optimizer, train_criterion}
  best.pth.tar (SSS, Google Drive) : legacy torch format, dict {start_it, stage, state_dict, optimizer}

Inputs expected by the embed functions:
  faces : list of JPG paths (face-centred, roughly square crops, any size)
  voices: list of WAV paths, 16 kHz mono (other rates are resampled), 3-180 s

All embed functions return float32 numpy arrays of shape (n, d).
"""

import os
import math
import urllib.request

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchaudio
from PIL import Image
from scipy.io import wavfile

CKPT_DIR = '/content/ckpt'
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
SR = 16000

# ----------------------------------------------------------------------------------
# generic helpers
# ----------------------------------------------------------------------------------

def _download(url, dst):
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        return dst
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    print(f'[download] {url} -> {dst}')
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as r, open(dst + '.part', 'wb') as f:
        while True:
            chunk = r.read(1 << 20)
            if not chunk:
                break
            f.write(chunk)
    os.replace(dst + '.part', dst)
    return dst


def _gdrive_download(file_id, dst):
    """gdown if available, otherwise the direct usercontent URL (works for public files)."""
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        return dst
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    try:
        import gdown  # noqa
        gdown.download(id=file_id, output=dst, quiet=False)
        if os.path.exists(dst) and os.path.getsize(dst) > 0:
            return dst
    except Exception as e:  # pragma: no cover
        print(f'[gdown failed: {e}] falling back to direct URL')
    return _download(f'https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t', dst)


def _torch_load(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except Exception:
        return torch.load(path, map_location='cpu', weights_only=False)


def _report_load(name, result):
    """Print the IncompatibleKeys of a strict=False load; return (missing, unexpected)."""
    print(f'[{name}] load_state_dict: missing={list(result.missing_keys)} '
          f'unexpected={list(result.unexpected_keys)}')
    return list(result.missing_keys), list(result.unexpected_keys)


def load_wav_16k(path):
    """-> float32 numpy in [-1, 1], mono, 16 kHz.
    int16 /32768 mirrors VisualVoice data/audioVisual_dataset.py L158-163 and
    librosa.load (CMPC dataloader.py L49). Stereo -> channel mean. Resampled if needed."""
    sr, x = wavfile.read(path)
    if x.dtype == np.int16:
        x = x.astype(np.float32) / 32768.0
    elif x.dtype == np.int32:
        x = x.astype(np.float32) / 2147483648.0
    elif x.dtype == np.uint8:
        x = (x.astype(np.float32) - 128.0) / 128.0
    else:
        x = x.astype(np.float32)
    if x.ndim > 1:
        x = x.mean(axis=1)
    if sr != SR:
        x = torchaudio.functional.resample(torch.from_numpy(x), sr, SR).numpy()
    return np.ascontiguousarray(x, dtype=np.float32)


def _window_starts(n, win, tail_min_frac=0.5):
    """Non-overlapping windows [0,W),[W,2W),... over a clip of n samples/frames.
    If the trailing remainder is >= tail_min_frac*W a final window ending at n is added
    (same idea as VisualVoice test.py L236-239 'deal with the last segment')."""
    if n <= win:
        return [0]
    starts = list(range(0, n - win + 1, win))
    rem = n - (starts[-1] + win)
    if rem >= tail_min_frac * win:
        starts.append(n - win)
    return starts


def _run_windows(fn, clips, max_windows):
    """clips: list (per clip) of list of window tensors with identical shape.
    Runs fn on chunks of <= max_windows windows, returns per-clip mean of fn outputs (torch, cpu)."""
    owner, flat = [], []
    for ci, ws in enumerate(clips):
        for w in ws:
            owner.append(ci)
            flat.append(w)
    outs = []
    with torch.no_grad():
        for i in range(0, len(flat), max_windows):
            x = torch.stack(flat[i:i + max_windows]).to(DEV, non_blocking=True)
            outs.append(fn(x).float().cpu())
    outs = torch.cat(outs, 0)
    owner = torch.tensor(owner)
    n = len(clips)
    acc = torch.zeros(n, outs.shape[1]).index_add_(0, owner, outs)
    cnt = torch.zeros(n).index_add_(0, owner, torch.ones(len(owner)))
    return acc / cnt.unsqueeze(1)


# ==================================================================================
# A. VisualVoice cross-modal pretraining
# ==================================================================================
VV_URLS = {
    'facial': 'http://dl.fbaipublicfiles.com/VisualVoice/cross-modal-pretraining/facial.pth',
    'vocal': 'http://dl.fbaipublicfiles.com/VisualVoice/cross-modal-pretraining/vocal.pth',
}
VV_AUDIO_LEN = 2.55          # options/base_options.py L32 (--audio_length 2.55, README train cmd L146)
VV_WIN = int(VV_AUDIO_LEN * SR)   # 40800 samples; data/audioVisual_dataset.py L115
VV_NFFT, VV_HOP, VV_WINLEN = 512, 160, 400   # base_options.py L34-36; README L147-149
VV_FEAT_DIM = 128            # README L164 (--identity_feature_dim 128); checkpoint fc.weight is (128,512)


class VVResnet18(nn.Module):
    """models/networks.py L74-106 (class Resnet18). Built on torchvision resnet18 exactly like
    models/models.py L20-23 (build_facial) and L66-69 (build_vocal):
      conv1 is replaced to accept `input_channel` channels and is ALSO element 0 of
      feature_extraction, so the state_dict has both 'conv1.weight' and 'feature_extraction.0.weight'
      (verified in facial.pth / vocal.pth). with_fc=True because identity_feature_dim=128 != 512
      (train.py / test.py: 'if opt.identity_feature_dim != 512: opt.with_fc = True')."""

    def __init__(self, input_channel=3, pool_type='maxpool', fc_in=512, fc_out=VV_FEAT_DIM):
        super().__init__()
        original_resnet = torchvision.models.resnet18(weights=None)
        self.pool_type = pool_type
        self.conv1 = nn.Conv2d(input_channel, 64, kernel_size=7, stride=2, padding=3, bias=False)  # networks.py L82
        layers = [self.conv1]
        layers.extend(list(original_resnet.children())[1:-2])        # networks.py L84: bn1, relu, maxpool, layer1-4
        self.feature_extraction = nn.Sequential(*layers)
        self.fc = nn.Linear(fc_in, fc_out)                            # networks.py L88

    def forward(self, x):                                             # networks.py L91-106
        x = self.feature_extraction(x)
        if self.pool_type == 'avgpool':
            x = F.adaptive_avg_pool2d(x, 1)
        else:                                                         # README: --visual_pool maxpool --audio_pool maxpool
            x = F.adaptive_max_pool2d(x, 1)
        x = x.view(x.size(0), -1)
        return self.fc(x)                                             # (B, 128); repo returns .view(B,-1,1,1)


_VV = {}


def load_vv():
    """Returns dict(face=VVResnet18(3ch), voice=VVResnet18(2ch)) in eval mode on DEV, strict-loaded."""
    if _VV:
        return _VV
    face = VVResnet18(input_channel=3)
    voice = VVResnet18(input_channel=2)   # train.py / test.py L106-111: build_vocal(input_channel=2)
    for name, net in (('facial', face), ('vocal', voice)):
        sd = _torch_load(_download(VV_URLS[name], os.path.join(CKPT_DIR, f'vv_{name}.pth')))
        # models/models.py L27-31 does a size-filtered partial load; here the checkpoint matches exactly:
        net.load_state_dict(sd, strict=True)
        print(f'[VisualVoice {name}] strict load OK ({len(sd)} tensors)')
        net.eval().to(DEV)
    _VV.update(face=face, voice=voice)
    return _VV


# face preprocessing: data/audioVisual_dataset.py L125-132: Resize(224), ToTensor, Normalize(ImageNet)
# (--normalization is set in the README train/test commands L155/L194). Resize(224) resizes the
# shorter side; crops here are ~square so we resize to exactly 224x224 to allow batching.
_VV_FACE_TF = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def vv_embed_faces(paths, bs=64, l2=False):
    """-> (n, 128) float32. l2=False returns the raw identity feature; the repo only L2-normalises when
    --l2_feature_normalization is passed (not in README commands) and its cross-modal triplet loss is
    cosine-based (models/criterion.py L75-89, train.py L187), so cosine similarity is scale-free anyway."""
    net = load_vv()['face']
    out = []
    with torch.no_grad():
        for i in range(0, len(paths), bs):
            x = torch.stack([_VV_FACE_TF(Image.open(p).convert('RGB')) for p in paths[i:i + bs]]).to(DEV)
            e = net(x)
            if l2:
                e = F.normalize(e, p=2, dim=1)
            out.append(e.float().cpu())
    return torch.cat(out).numpy().astype(np.float32)


_VV_WINDOW = None


def vv_spectrogram(seg):
    """seg: float32 numpy (40800,) -> torch (2, 256, 256).
    data/audioVisual_dataset.py L90-95 generate_spectrogram_complex:
      librosa.stft(audio, hop_length=160, n_fft=512, win_length=400, center=True)  (librosa 0.7.0 defaults:
      periodic Hann window, reflect padding) -> channels (real, imag), shape (2, 257, 256).
    The vocal net is applied to audio_spec[:, :, :-1, :] (models/audioVisual_model.py, net_classifier calls),
    i.e. the last frequency bin (257th) is dropped -> (2, 256, 256)."""
    global _VV_WINDOW
    if _VV_WINDOW is None:
        _VV_WINDOW = torch.hann_window(VV_WINLEN, periodic=True)
    s = torch.stft(torch.from_numpy(seg), n_fft=VV_NFFT, hop_length=VV_HOP, win_length=VV_WINLEN,
                   window=_VV_WINDOW, center=True, pad_mode='reflect', return_complex=True)
    spec = torch.stack([s.real, s.imag], 0)        # (2, 257, T)
    return spec[:, :-1, :].float()                  # (2, 256, 256)


def _vv_rms_normalize(samples, desired_rms=0.1, eps=1e-4):
    """data/audioVisual_dataset.py L26-29 `normalize` (applied because README uses --audio_normalization L165)."""
    rms = np.maximum(eps, np.sqrt(np.mean(samples ** 2)))
    return samples * (desired_rms / rms)


def vv_voice_windows(wav):
    """Split a full clip into 2.55 s non-overlapping windows (plus a tail window), RMS-normalise each
    window (dataset normalises per 2.55 s sample), return list of (2,256,256) tensors.
    Clips shorter than 2.55 s are zero-padded at the end (the repo simply skips such clips,
    dataset L165-166; padding rule is ours)."""
    if len(wav) < VV_WIN:
        wav = np.pad(wav, (0, VV_WIN - len(wav)))
    ws = []
    for st in _window_starts(len(wav), VV_WIN):
        seg = np.clip(wav[st:st + VV_WIN], -1.0, 1.0)       # test.py L32-35 clip_audio
        seg = _vv_rms_normalize(seg)
        ws.append(vv_spectrogram(seg.astype(np.float32)))
    return ws


def vv_embed_voices(paths, bs=32, max_windows=128, l2=False):
    """-> (n, 128) float32. Each clip is cut into consecutive non-overlapping 2.55 s windows (the
    pretraining segment length); window embeddings are mean-pooled per clip (our choice: the repo
    only ever embeds single 2.55 s segments). max_windows caps GPU work per forward (2x256x256 inputs)."""
    net = load_vv()['voice']
    out = []
    for i in range(0, len(paths), bs):
        clips = [vv_voice_windows(load_wav_16k(p)) for p in paths[i:i + bs]]
        e = _run_windows(net, clips, max_windows)
        if l2:
            e = F.normalize(e, p=2, dim=1)
        out.append(e)
    return torch.cat(out).numpy().astype(np.float32)


# ==================================================================================
# B. CMPC (Cocoxili/CMPC, IJCAI 2022)
# ==================================================================================
CMPC_URLS = {
    'CMPC': 'https://github.com/Cocoxili/CMPC/releases/download/v1.0.0/checkpoint_CMPC.pth.tar',  # 511,192,079 bytes
    'CID': 'https://github.com/Cocoxili/CMPC/releases/download/v1.0.0/checkpoint_CID.pth.tar',    # 511,192,079 bytes
}
# experiments/cmpc/CONFIG.yaml L36-42: sampling_rate 16000, fbank_nframes 500, frame_width 100 ms,
# frame_shift 10 ms, n_mels 64, face_size [224,224]; data_transform.py L81-82: n_fft = 1600, hop = 160
CMPC_NFFT, CMPC_HOP, CMPC_NMELS, CMPC_NFRAMES = 1600, 160, 64, 500
CMPC_DIM = 512   # CONFIG.yaml L69 last_dim 512; fv_resnet.py L358 fc = ChannelPooling(512) == identity on 512-d


def _cmpc_resnet34(input_channel):
    """experiments/cmpc/models/fv_resnet.py L349-359 resnet34(): torchvision-identical ResNet(BasicBlock,[3,4,6,3])
    (fv_resnet.py L226-333 is a verbatim copy of torchvision's ResNet, same parameter names), then
    conv1 replaced by Conv2d(input_channel, 64, 7, 2, 3, bias=False) and fc = ChannelPooling(out_dim).
    ChannelPooling (L375-381) is AdaptiveAvgPool1d(512) applied to a 512-d vector -> identity, no params."""
    m = torchvision.models.resnet34(weights=None)
    m.conv1 = nn.Conv2d(input_channel, 64, kernel_size=7, stride=2, padding=3, bias=False)
    m.fc = nn.Identity()
    return m


class CMPC_FV_Wrapper(nn.Module):
    """experiments/cmpc/models/fv_wrapper.py L20-41: voice_subnet = resnet34(1ch), face_subnet = resnet34(3ch).
    (proj head is commented out in the repo; checkpoint contains only face_subnet.* and voice_subnet.*)"""

    def __init__(self):
        super().__init__()
        self.voice_subnet = _cmpc_resnet34(1)
        self.face_subnet = _cmpc_resnet34(3)

    def forward(self, audio, frame):
        return self.voice_subnet(audio), self.face_subnet(frame)


_CMPC = {}


def load_cmpc(which='CMPC'):
    """Downloads checkpoint_CMPC.pth.tar (the paper's full method; CID is the ablation without prototype
    contrast). verification.py L84-93 load_model: ckp = torch.load(fn); model.load_state_dict(ckp['model']).
    Verified keys: ckp = {'epoch': 768000, 'model': OrderedDict(432 tensors, prefixes face_subnet./voice_subnet.,
    no 'module.'), 'optimizer': {...}, 'train_criterion': {}}."""
    if which in _CMPC:
        return _CMPC[which]
    ckp = _torch_load(_download(CMPC_URLS[which], os.path.join(CKPT_DIR, f'checkpoint_{which}.pth.tar')))
    sd = ckp['model']
    sd = {k.replace('module.', '', 1) if k.startswith('module.') else k: v for k, v in sd.items()}
    model = CMPC_FV_Wrapper()
    model.load_state_dict(sd, strict=True)
    print(f'[CMPC {which}] strict load OK ({len(sd)} tensors, epoch={ckp.get("epoch")})')
    model.eval().to(DEV)
    _CMPC[which] = model
    return model


def cmpc_load_face(path):
    """experiments/cmpc/dataloader.py L80-84 load_face: PIL RGB .resize([224,224]) (Pillow 9.0.1 default
    resample = BICUBIC), CHW, (x - 127.5) / 127.5."""
    face = Image.open(path).convert('RGB').resize((224, 224), Image.BICUBIC)
    face = np.transpose(np.array(face), (2, 0, 1))
    return torch.from_numpy(((face - 127.5) / 127.5).astype('float32'))


def cmpc_embed_faces(paths, bs=64, l2=True):
    """-> (n, 512) float32, L2-normalised (criterions/cmpc.py L21-22 and matching.py L94 use
    F.normalize / cosine similarity on the raw 512-d outputs)."""
    model = load_cmpc()
    out = []
    with torch.no_grad():
        for i in range(0, len(paths), bs):
            x = torch.stack([cmpc_load_face(p) for p in paths[i:i + bs]]).to(DEV)
            e = model.face_subnet(x)
            if l2:
                e = F.normalize(e, p=2, dim=1)
            out.append(e.float().cpu())
    return torch.cat(out).numpy().astype(np.float32)


_CMPC_MEL = None


def cmpc_logmel(wav):
    """experiments/cmpc/data_transform.py L81-90 / dataloader.py L69-77:
      librosa.feature.melspectrogram(y, 16000, n_fft=1600, hop_length=160, n_mels=64)   (librosa 0.6.2 defaults:
      power=2, periodic Hann, center=True, reflect pad, fmin=0, fmax=8000, slaney mel scale + slaney norm)
      logmel = librosa.power_to_db(melspec)  -> 10*log10(max(S,1e-10)), then clipped to (max - 80 dB),
      computed on the WHOLE utterance before any cropping (data_transform.py runs on full wav files).
    Returns torch (64, T) float32."""
    global _CMPC_MEL
    if _CMPC_MEL is None:
        _CMPC_MEL = torchaudio.transforms.MelSpectrogram(
            sample_rate=SR, n_fft=CMPC_NFFT, win_length=CMPC_NFFT, hop_length=CMPC_HOP,
            f_min=0.0, f_max=SR / 2, n_mels=CMPC_NMELS, window_fn=torch.hann_window, power=2.0,
            center=True, pad_mode='reflect', norm='slaney', mel_scale='slaney')
    S = _CMPC_MEL(torch.from_numpy(wav))
    logmel = 10.0 * torch.log10(torch.clamp(S, min=1e-10))
    logmel = torch.maximum(logmel, logmel.max() - 80.0)     # power_to_db top_db=80
    return logmel.float()


def cmpc_voice_windows(wav):
    """500-frame (5 s) windows of the log-mel, non-overlapping + tail window, each (1, 64, 500).
    Shorter clips: zero-pad the log-mel symmetrically, exactly dataloader.py L91-93 load_voice."""
    lm = cmpc_logmel(wav)
    if lm.shape[1] < CMPC_NFRAMES:
        shortage = math.floor((CMPC_NFRAMES - lm.shape[1] + 1) / 2)
        lm = F.pad(lm, (shortage, shortage))
    return [lm[:, st:st + CMPC_NFRAMES].unsqueeze(0) for st in _window_starts(lm.shape[1], CMPC_NFRAMES)]


def cmpc_embed_voices(paths, bs=32, max_windows=256, l2=True):
    """-> (n, 512) float32. Mean of per-window (5 s) embeddings, then L2 (verification.py L70-76 averages
    several 500-frame crops per utterance before cosine scoring, so mean-then-normalise mirrors the repo)."""
    model = load_cmpc()
    out = []
    for i in range(0, len(paths), bs):
        clips = [cmpc_voice_windows(load_wav_16k(p)) for p in paths[i:i + bs]]
        e = _run_windows(model.voice_subnet, clips, max_windows)
        if l2:
            e = F.normalize(e, p=2, dim=1)
        out.append(e)
    return torch.cat(out).numpy().astype(np.float32)


# ==================================================================================
# C. Seeking the Shape of Sound (KID-7391/seeking-the-shape-of-sound, master)
# ==================================================================================
SSS_GDRIVE_ID = '1ZCPMk_0kKz8YO37ciAVJRDnmnTCqNhoG'   # README "pretrained evaluation model" -> best.pth.tar
SSS_DIM = 128                                         # config/train_main.yaml L8/L15 output_channel 128
SSS_NUM_CLASSES = 924                                 # config/train_main.yaml L9/L16
SSS_AUDIO_LEN = SR * 10                               # dataloaders/base_dataset.py L36-40: first 10 s, tiled if shorter


class SSS_NormalizeConv(nn.Module):
    """models/nn/module.py L460-483: fixed depthwise 1x1 conv implementing (x - mean) / std with
    mean = std = 127.5 (config/train_main.yaml L29-36). Weights live in the checkpoint ('normalize.conv.*')."""

    def __init__(self, in_channel=3, normal_mean=(127.5, 127.5, 127.5), normal_std=(127.5, 127.5, 127.5)):
        super().__init__()
        bias = np.array([-m / s for m, s in zip(normal_mean, normal_std)])
        weight = np.array([[[[1.0 / s for s in normal_std]]]]).transpose((3, 0, 1, 2))
        self.conv = nn.Conv2d(in_channel, in_channel, 1, stride=1, padding=0, groups=in_channel, bias=True)
        self.conv.weight = nn.Parameter(torch.Tensor(weight), requires_grad=False)
        self.conv.bias = nn.Parameter(torch.Tensor(bias), requires_grad=False)

    def forward(self, x):
        return self.conv(x)


# ---- face backbone: IR-50 (models/nn/ir_se_model.py). backbones.py L16 calls IR_50([112,112], ...) which is
#      Backbone(input_size, 50, 'ir') -> plain bottleneck_IR units (no SE), checkpoint confirms (no res_layer.5).
class SSS_Flatten(nn.Module):
    def forward(self, x):
        return x.view(x.size(0), -1)


class SSS_bottleneck_IR(nn.Module):              # ir_se_model.py L49-66
    def __init__(self, in_channel, depth, stride):
        super().__init__()
        if in_channel == depth:
            self.shortcut_layer = nn.MaxPool2d(1, stride)
        else:
            self.shortcut_layer = nn.Sequential(nn.Conv2d(in_channel, depth, (1, 1), stride, bias=False),
                                                nn.BatchNorm2d(depth))
        self.res_layer = nn.Sequential(
            nn.BatchNorm2d(in_channel),
            nn.Conv2d(in_channel, depth, (3, 3), (1, 1), 1, bias=False), nn.PReLU(depth),
            nn.Conv2d(depth, depth, (3, 3), stride, 1, bias=False), nn.BatchNorm2d(depth))

    def forward(self, x):
        return self.res_layer(x) + self.shortcut_layer(x)


def _sss_ir50_blocks():                           # ir_se_model.py L98-109
    def get_block(in_channel, depth, num_units, stride=2):
        return [(in_channel, depth, stride)] + [(depth, depth, 1) for _ in range(num_units - 1)]
    return [get_block(64, 64, 3), get_block(64, 128, 4), get_block(128, 256, 14), get_block(256, 512, 3)]


class SSS_IR50(nn.Module):                        # ir_se_model.py L129-172 Backbone(input_size=[112,112], 50, 'ir')
    def __init__(self):
        super().__init__()
        self.input_layer = nn.Sequential(nn.Conv2d(3, 64, (3, 3), 1, 1, bias=False), nn.BatchNorm2d(64), nn.PReLU(64))
        self.output_layer = nn.Sequential(nn.BatchNorm2d(512), nn.Dropout(), SSS_Flatten(),
                                          nn.Linear(512 * 7 * 7, 512), nn.BatchNorm1d(512))   # L143-148 (112 input)
        self.body = nn.Sequential(*[SSS_bottleneck_IR(i, d, s) for blk in _sss_ir50_blocks() for (i, d, s) in blk])

    def forward(self, x):
        return self.output_layer(self.body(self.input_layer(x)))


# ---- voice backbone: thin ResNetSE34 (models/nn/res_se_34l_model.py + models/nn/ResNetBlocks.py)
class SSS_SELayer(nn.Module):                     # ResNetBlocks.py L81-95, reduction 8
    def __init__(self, channel, reduction=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Linear(channel, channel // reduction), nn.ReLU(inplace=True),
                                nn.Linear(channel // reduction, channel), nn.Sigmoid())

    def forward(self, x):
        b, c, _, _ = x.size()
        return x * self.fc(self.avg_pool(x).view(b, c)).view(b, c, 1, 1)


class SSS_SEBasicBlock(nn.Module):                # ResNetBlocks.py L7-37 (note relu BEFORE bn1, as in the repo)
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None, reduction=8):
        super().__init__()
        self.conv1 = nn.Conv2d(inplanes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.se = SSS_SELayer(planes, reduction)
        self.downsample = downsample

    def forward(self, x):
        residual = x
        out = self.bn1(self.relu(self.conv1(x)))
        out = self.se(self.bn2(self.conv2(out)))
        if self.downsample is not None:
            residual = self.downsample(x)
        return self.relu(out + residual)


class SSS_ResNetSE34(nn.Module):
    """res_se_34l_model.py L11-122: ResNetSE(SEBasicBlock, [3,4,6,3], [16,32,64,128], nOut=512, encoder 'SAP',
    n_mels=40, log_input=True). Front-end (L33): torchaudio MelSpectrogram(16000, n_fft 512, win 400, hop 160,
    hamming window, n_mels 40) on the RAW waveform, +1e-6, log, InstanceNorm1d(40)."""

    def __init__(self, layers=(3, 4, 6, 3), num_filters=(16, 32, 64, 128), nOut=512, n_mels=40):
        super().__init__()
        self.inplanes = num_filters[0]
        self.conv1 = nn.Conv2d(1, num_filters[0], kernel_size=7, stride=(2, 1), padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(num_filters[0])
        self.relu = nn.ReLU(inplace=True)
        self.layer1 = self._make_layer(num_filters[0], layers[0])
        self.layer2 = self._make_layer(num_filters[1], layers[1], stride=(2, 2))
        self.layer3 = self._make_layer(num_filters[2], layers[2], stride=(2, 2))
        self.layer4 = self._make_layer(num_filters[3], layers[3], stride=(1, 1))
        self.instancenorm = nn.InstanceNorm1d(n_mels)
        self.torchfb = torchaudio.transforms.MelSpectrogram(sample_rate=16000, n_fft=512, win_length=400,
                                                            hop_length=160, window_fn=torch.hamming_window,
                                                            n_mels=n_mels)
        self.sap_linear = nn.Linear(num_filters[3], num_filters[3])
        self.attention = nn.Parameter(torch.FloatTensor(num_filters[3], 1))
        self.fc = nn.Linear(num_filters[3], nOut)

    def _make_layer(self, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes:
            downsample = nn.Sequential(nn.Conv2d(self.inplanes, planes, kernel_size=1, stride=stride, bias=False),
                                       nn.BatchNorm2d(planes))
        layers = [SSS_SEBasicBlock(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes
        layers += [SSS_SEBasicBlock(self.inplanes, planes) for _ in range(1, blocks)]
        return nn.Sequential(*layers)

    def forward(self, x):                          # res_se_34l_model.py L77-111
        x = self.torchfb(x) + 1e-6
        x = x.log()
        x = self.instancenorm(x).unsqueeze(1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        x = torch.mean(x, dim=2, keepdim=True)
        x = x.permute(0, 3, 1, 2).squeeze(-1)       # (B, T, C)
        h = torch.tanh(self.sap_linear(x))
        w = torch.matmul(h, self.attention).squeeze(dim=2)
        w = F.softmax(w, dim=1).view(x.size(0), x.size(1), 1)
        x = torch.sum(x * w, dim=1)
        return self.fc(x.view(x.size(0), -1))


class SSS_Classifier(nn.Module):                  # backbones.py L48-52 (only `weight` exists in released code)
    def __init__(self, nin, nout):
        super().__init__()
        self.weight = nn.Parameter(torch.FloatTensor(nout, nin))


class SSS_FaceNet(nn.Module):                     # backbones.py L12-28 SEResNet50IR (model + fc + cls)
    def __init__(self, output_channel=SSS_DIM, num_classes=SSS_NUM_CLASSES):
        super().__init__()
        self.model = SSS_IR50()
        self.fc = nn.Sequential(nn.BatchNorm1d(512), nn.ReLU(inplace=True),
                                nn.Linear(512, output_channel, bias=False), nn.Dropout(0.5))
        self.cls = SSS_Classifier(output_channel, num_classes)

    def forward(self, x):
        return self.fc(self.model(x))


class SSS_VoiceNet(nn.Module):                    # backbones.py L30-45 ThinResNet34 (model + fc)
    def __init__(self, output_channel=SSS_DIM):
        super().__init__()
        self.model = SSS_ResNetSE34()
        self.fc = nn.Sequential(nn.BatchNorm1d(512), nn.ReLU(inplace=True),
                                nn.Linear(512, output_channel, bias=False), nn.Dropout(0.5))

    def forward(self, x):
        return self.fc(self.model(x))


class SSS_Center(nn.Module):                      # models/center_model.py L17-20 (two frozen vectors)
    def __init__(self, num_classes=SSS_NUM_CLASSES):
        super().__init__()
        self.loss_mean = nn.Parameter(torch.full((num_classes,), 14.0), requires_grad=False)
        self.weight = nn.Parameter(torch.ones(num_classes), requires_grad=False)


class SSS_Voice2Face(nn.Module):
    """models/voice2face_model.py L29-39 + base_model.py L15-16: submodules named normalize, V, F, center."""

    def __init__(self):
        super().__init__()
        self.normalize = SSS_NormalizeConv()
        self.V = SSS_VoiceNet()
        self.F = SSS_FaceNet()
        self.center = SSS_Center()


_SSS = {}


def load_sss():
    """Downloads best.pth.tar from Google Drive (id 1ZCPMk_0kKz8YO37ciAVJRDnmnTCqNhoG). Verified structure:
    legacy torch pickle, dict {start_it: 8000, stage: 0, state_dict: OrderedDict(655 tensors, prefixes
    normalize./V./F./center., no 'module.'), optimizer: {...}}. utils/utils.py load_checkpoint strips a
    'module.' prefix and load_pretrained_model loads with strict=False. The checkpoint carries two extra
    buffers F.cls.vote (2,924,2) and F.cls.score (2,924) that are absent from the released Classifier
    (backbones.py L48-52) and unused at inference; they are the only expected 'unexpected' keys."""
    if _SSS:
        return _SSS['model']
    ckp = _torch_load(_gdrive_download(SSS_GDRIVE_ID, os.path.join(CKPT_DIR, 'sss_best.pth.tar')))
    sd = ckp['state_dict'] if 'state_dict' in ckp else ckp
    sd = {k.replace('module.', '', 1) if k.startswith('module.') else k: v for k, v in sd.items()}
    model = SSS_Voice2Face()
    res = model.load_state_dict(sd, strict=False)
    missing, unexpected = _report_load('SSS', res)
    assert not missing, f'SSS: missing keys {missing}'
    assert set(unexpected) <= {'F.cls.vote', 'F.cls.score'}, f'SSS: unexpected keys {unexpected}'
    model.eval().to(DEV)
    _SSS['model'] = model
    return model


def sss_load_face(path):
    """dataloaders/base_dataset.py L16-28 pil_loader: PIL RGB -> uint8 -> BGR; custom_transforms.py L96-127
    Resize(112,112) with cv2.INTER_LINEAR (PIL BILINEAR used here; crops are ~square so output is 112x112);
    ToTensor (L8-31): CHW float, still in 0..255. The model's `normalize` layer then maps to [-1,1]."""
    img = Image.open(path).convert('RGB').resize((112, 112), Image.BILINEAR)
    img = np.array(img).astype(np.uint8)[:, :, ::-1]           # BGR
    return torch.from_numpy(img.transpose((2, 0, 1)).copy()).float()


def sss_embed_faces(paths, bs=64, l2=True):
    """-> (n, 128) float32. eval.py L36-38: F(normalize(imgs)); scoring uses cosine_distance (L168-169), i.e.
    F.normalize on both sides, so L2-normalised outputs are returned by default."""
    model = load_sss()
    out = []
    with torch.no_grad():
        for i in range(0, len(paths), bs):
            x = torch.stack([sss_load_face(p) for p in paths[i:i + bs]]).to(DEV)
            e = model.F(model.normalize(x))
            if l2:
                e = F.normalize(e, p=2, dim=1)
            out.append(e.float().cpu())
    return torch.cat(out).numpy().astype(np.float32)


def sss_voice_windows(wav):
    """base_dataset.py L29-41: raw int16 sample values (NOT scaled to [-1,1]) as float32, tiled to >= 10 s,
    first 10 s used at evaluation. We feed int16-scale values (x * 32768) and take consecutive 10 s windows
    (+ tail) over the whole clip, mean-pooling (the repo embeds only the first 10 s). Note log + InstanceNorm
    in the front-end make the result scale-invariant up to the +1e-6 epsilon."""
    x = wav * 32768.0
    if len(x) < SSS_AUDIO_LEN:
        x = np.tile(x, (SSS_AUDIO_LEN + len(x) - 1) // len(x))
    return [torch.from_numpy(x[st:st + SSS_AUDIO_LEN].astype(np.float32))
            for st in _window_starts(len(x), SSS_AUDIO_LEN)]


def sss_embed_voices(paths, bs=32, max_windows=64, l2=True):
    """-> (n, 128) float32, mean over 10 s windows then L2 (cosine scoring in eval.py)."""
    model = load_sss()
    out = []
    for i in range(0, len(paths), bs):
        clips = [sss_voice_windows(load_wav_16k(p)) for p in paths[i:i + bs]]
        e = _run_windows(model.V, clips, max_windows)
        if l2:
            e = F.normalize(e, p=2, dim=1)
        out.append(e)
    return torch.cat(out).numpy().astype(np.float32)


# ==================================================================================
# self-test
# ==================================================================================
def _selftest(download=True):
    torch.manual_seed(0)
    print('device:', DEV, '| torch', torch.__version__, '| torchaudio', torchaudio.__version__)

    # --- shape tests on random inputs (no downloads) ---
    with torch.no_grad():
        f = VVResnet18(3).eval(); v = VVResnet18(2).eval()
        print('VV  face  (2,3,224,224) ->', tuple(f(torch.randn(2, 3, 224, 224)).shape))
        spec = vv_spectrogram(np.random.randn(VV_WIN).astype(np.float32))
        print('VV  spec  (40800,)      ->', tuple(spec.shape))
        print('VV  voice (2,2,256,256) ->', tuple(v(torch.stack([spec, spec])).shape))

        m = CMPC_FV_Wrapper().eval()
        lm = cmpc_logmel(np.random.randn(SR * 5).astype(np.float32) * 0.1)
        print('CMPC logmel (5 s)       ->', tuple(lm.shape))
        a, b = m(torch.randn(2, 1, 64, 500), torch.randn(2, 3, 224, 224))
        print('CMPC voice/face         ->', tuple(a.shape), tuple(b.shape))

        s = SSS_Voice2Face().eval()
        print('SSS face  (2,3,112,112) ->', tuple(s.F(s.normalize(torch.rand(2, 3, 112, 112) * 255)).shape))
        print('SSS voice (2,160000)    ->', tuple(s.V(torch.randn(2, SSS_AUDIO_LEN) * 3000).shape))

    if not download:
        return
    # --- checkpoint loads (strict where possible) ---
    vv = load_vv()
    assert next(vv['face'].parameters()).device.type == DEV.split(':')[0]
    cm = load_cmpc()
    ss = load_sss()
    with torch.no_grad():
        e1 = vv['face'](torch.randn(1, 3, 224, 224, device=DEV))
        e2 = cm.face_subnet(torch.randn(1, 3, 224, 224, device=DEV))
        e3 = ss.F(ss.normalize(torch.rand(1, 3, 112, 112, device=DEV) * 255))
    print('loaded-model output dims: VV', e1.shape[1], '| CMPC', e2.shape[1], '| SSS', e3.shape[1])
    print('self-test OK')



# NB42: read every WAV with librosa (24-bit and float files included), 16 kHz mono; overrides the scipy reader above
def load_wav_16k(path):
    return np.ascontiguousarray(librosa.load(path, sr=16000, mono=True)[0], dtype=np.float32)


# ---------------------------------------------------------------------------------------------------------
# NB42 extraction: every block on every row of the three splits, resumable per (block, split) -> features/xmodal.npz
# ---------------------------------------------------------------------------------------------------------
CKPT_DIR = os.path.join(CACHE, 'xmodal_ckpt'); os.makedirs(CKPT_DIR, exist_ok=True)   # checkpoints cached on Drive (1.5 GB)
del CLIPM, WFE, WENC; torch.cuda.empty_cache()
MODELS = {}
for name, loader in [('vv', load_vv), ('cmpc', load_cmpc), ('sss', load_sss)]:
    try:
        loader(); MODELS[name] = True; enote('%s: checkpoint loaded' % name)
    except Exception as e:
        MODELS[name] = False; enote('%s: NOT loaded (%s) -> skipped' % (name, repr(e)[:160]))
assert MODELS['vv'] or MODELS['cmpc'], 'neither VisualVoice nor CMPC could be loaded - tell Claude'
EMBED = {'vv_face': vv_embed_faces, 'vv_voice': vv_embed_voices, 'cmpc_face': cmpc_embed_faces, 'cmpc_voice': cmpc_embed_voices, 'sss_face': sss_embed_faces, 'sss_voice': sss_embed_voices}
BLOCKS = [b for b in EMBED if MODELS[b.split('_')[0]]]
XM = {}
t_all = time.time()
for b in BLOCKS:
    for s in SPLITS:
        pf = os.path.join(PARTS, '%s_%s.npy' % (b, s))
        if os.path.exists(pf):
            XM[(b, s)] = np.load(pf); continue
        t0 = time.time(); paths = FACES[s] if b.endswith('face') else WAVS[s]; out = []
        for i in tqdm(range(0, len(paths), 256), desc='%s %s' % (b, s)):
            out.append(EMBED[b](paths[i:i + 256]))
        E = np.concatenate(out).astype(np.float32); assert E.shape[0] == len(paths) and np.isfinite(E).all(), '%s %s: bad output - tell Claude' % (b, s)
        np.save(pf, E); XM[(b, s)] = E; enote('%s %s: %d rows x %d-d in %.0f min' % (b, s, E.shape[0], E.shape[1], (time.time() - t0) / 60))
np.savez(os.path.join(FEATDIR, 'xmodal.npz'), **{'%s_%s' % k: v for k, v in XM.items()})
enote('extraction done in %.0f min -> features/xmodal.npz (%s)' % ((time.time() - t_all) / 60, ', '.join(BLOCKS)))
# ---- zero-training check on challenge rows: each model's own face-voice cosine on random and same-gender pairs ----
Y_CH = np.array([int(t[5]) for t in lines]); cls2spk = {int(l[5]): l[4] for l in lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower())); g_of = {c: spk2g[cls2spk[c]] for c in np.unique(Y_CH)}
from sklearn.metrics import roc_curve
def eer_(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr; i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
def pairs_ch(n=3000, seed=0, same_gender=False):
    rng = np.random.default_rng(seed); spk_ = np.unique(Y_CH); rows_ = {s: np.where(Y_CH == s)[0] for s in spk_}; V, Fi, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); Fi.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_); cand = [t for t in spk_ if t != s and (not same_gender or g_of[t] == g_of[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); Fi.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(Fi), np.array(L)
def l2n(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
P_STD, P_GEN = pairs_ch(seed=0), pairs_ch(seed=1, same_gender=True)
for m in ['vv', 'cmpc', 'sss']:
    if not MODELS[m]:
        continue
    Fm, Vm = l2n(XM[(m + '_face', 'train')]), l2n(XM[(m + '_voice', 'train')])
    res = []
    for nm, (Vi, Fi, L) in [('random', P_STD), ('same-gender', P_GEN)]:
        res.append('%s %.1f' % (nm, eer_(L, (Fm[Fi] * Vm[Vi]).sum(1))))
    # identity gap inside each modality (same - different cosine) tells whether the block carries identity at all
    def gap(E, lab, n=800, seed=0):
        r = np.random.default_rng(seed); idx = r.choice(len(lab), n, replace=False); Z = l2n(E[idx]); C = Z @ Z.T
        same = (lab[idx][:, None] == lab[idx][None, :]) & ~np.eye(n, dtype=bool); diff = lab[idx][:, None] != lab[idx][None, :]
        return C[same].mean() - C[diff].mean()
    enote('%-4s zero-training cross-modal EER on challenge rows (its own cosine, no head): %s | identity gap face %.3f voice %.3f' % (m, ', '.join(res), gap(Fm, Y_CH), gap(Vm, Y_CH)))
enote('(reference: chance = 50; our trained head reaches about 18 random / 28 same-gender on the same kind of pairs in CV)')
NOTES42 = list(EXTRACT_NOTES)


In [ ]:
# Cell 4 - features, labels, CV folds, harness (NB36 Cell 2 verbatim, same as NB39 Cell 5)
import os, re, json, time, zipfile, copy
import numpy as np, pandas as pd
import torch.nn as nn, torch.nn.functional as Fn
from sklearn.metrics import roc_curve
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027'); FEATDIR = os.path.join(PROJECT, 'features'); RESDIR = os.path.join(PROJECT, 'results')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
LISTS = '/content/flag_lists'
if not os.path.exists(LISTS + '/done'):
    with zipfile.ZipFile(os.path.join(ZIP_DIR, 'train_set.zip')) as zf:
        zf.extractall(LISTS, members=[m for m in zf.namelist() if m.endswith(('.txt', '.csv'))])
    open(LISTS + '/done', 'w').close()
TXT = [os.path.join(r, f) for r, _, fs in os.walk(LISTS) for f in fs if f.endswith(('.txt', '.csv'))]
def find_one(*parts):
    hits = [p for p in TXT if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
Xv, y = read_feat(find_one('train_English_voices.csv'), 192); y = y.astype(int)
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y); id_gender = {c: spk2g[cls2spk[c]] for c in ids}
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz'))); EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz')))
ML = np.load(os.path.join(FEATDIR, 'mav_lang.npz')); EXT_SPK, EXT_LANG = ML['spk'], ML['lang']
assert list(EXT_SPK) == list(MAV['spk']) + list(EXTRA['spk']), 'mav_lang.npz does not match the feature files - tell Claude'
FEAT = {'ecapa': {'train': Xv, 'mav': MAV['ecapa'], 'extra': EXTRA['ecapa']}}
for name in ['clip', 'wavlm_sv', 'r293', 'redim', 'audeer', 'whisper']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in d.files if not s.endswith('_ag')}
FEAT['clip']['mav'], FEAT['clip']['extra'] = MAV['clip'], EXTRA['clip']; FEAT['wavlm_sv']['mav'], FEAT['wavlm_sv']['extra'] = MAV['wavlm_sv'], EXTRA['wavlm_sv']
FEAT['r293']['extra'] = EXTRA['r293']
for name in FEAT:
    for s in ['train', 'mav', 'extra']:
        assert s in FEAT[name] and len(FEAT[name][s]) == {'train': len(y), 'mav': len(MAV['spk']), 'extra': len(EXTRA['spk'])}[s], '%s/%s missing or wrong length - tell Claude' % (name, s)
N_CH = len(y); ext_spk = list(EXT_SPK)
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.r_[y, np.array([ext_lab[s] for s in ext_spk])]; D_ALL = np.r_[np.zeros(N_CH, int), np.ones(len(ext_spk), int)]; EXT = N_CH + np.arange(len(ext_spk))
def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def build(names, split):
    return np.concatenate([l2(FEAT[n][split]) for n in names], 1).astype(np.float32)
def build_train(names):
    return np.concatenate([build(names, 'train'), build(names, 'mav'), build(names, 'extra')])
NAP_K = 8
def nap_basis(V_ext, k):
    diffs = []
    for s in np.unique(EXT_SPK):
        m = EXT_SPK == s; langs = np.unique(EXT_LANG[m])
        if len(langs) < 2:
            continue
        mu = [V_ext[m & (EXT_LANG == l)].mean(0) for l in langs]
        for a in range(len(mu)):
            for b in range(a + 1, len(mu)):
                diffs.append(mu[a] - mu[b])
    Dm = np.stack(diffs); U = np.linalg.svd(Dm - Dm.mean(0), full_matrices=False)[2][:k]
    return U.astype(np.float32), len(diffs)
def nap_apply(X, U):
    return (X - (X @ U.T) @ U).astype(np.float32)
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed); spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5; rng = np.random.default_rng(42); folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))
_PROTO = {}
def max_sep_prototypes(n):
    if n in _PROTO:
        return _PROTO[n]
    P = np.array([[1.0, -1.0]])
    for k in range(2, n):
        Q = np.zeros((k, k + 1)); Q[0, 0] = 1.0; Q[0, 1:] = -1.0 / k; Q[1:, 1:] = np.sqrt(1 - 1.0 / k ** 2) * P; P = Q
    _PROTO[n] = torch.tensor(P.T, dtype=torch.float32); return _PROTO[n]
class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class - 1); self.register_buffer('P', max_sep_prototypes(n_class))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv); return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv); a = torch.sigmoid(self.att(torch.cat([f, v], 1))); fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused) @ self.P.t(), ft, vt
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t(); same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
def nce_loss(ft, vt, lab, tau):
    """symmetric identity-aware InfoNCE between the two scored embeddings: every same-identity row counts as a positive"""
    S = Fn.normalize(ft, dim=1) @ Fn.normalize(vt, dim=1).t() / tau; same = lab[:, None] == lab[None, :]
    def one(S_):
        logp = S_ - torch.logsumexp(S_, 1, keepdim=True); return -torch.logsumexp(logp.masked_fill(~same, -1e4), 1).mean()
    return 0.5 * (one(S) + one(S.t()))
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm; ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()
def dnorm(F_raw, V_raw, tr_idx):
    F, V = F_raw.copy(), V_raw.copy(); ch = None
    for d in np.unique(D_ALL[tr_idx]):
        trd = tr_idx[D_ALL[tr_idx] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_raw[trd].mean(0), F_raw[trd].std(0) + 1e-6, V_raw[trd].mean(0), V_raw[trd].std(0) + 1e-6)
        F[alld] = (F_raw[alld] - st[0]) / st[1]; V[alld] = (V_raw[alld] - st[2]) / st[3]
        if d == 0:
            ch = st
    return F, V, ch
SEEDS = [1, 2, 3]; P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS = 16, 8, 2.0, 1e-4, 100, 15, 60
def train_head(F_raw, V_raw, all_tr, vsets=None, seed=1, cfg=None, max_epochs=MAX_EP):
    """cfg keys: align in {None,'mse','nce'}, w (alignment weight), tau, repair (bool), p_in (input dropout)"""
    cfg = cfg or {}; align, w_al, tau, repair, p_in = cfg.get('align'), cfg.get('w', 1.0), cfg.get('tau', 0.07), cfg.get('repair', False), cfg.get('p_in', 0.0)
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev); model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), P_ID, replace=False)
            j_np = np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen])
            jv_np = np.concatenate([g[rng.permutation(K_CLIP)] for g in j_np.reshape(P_ID, K_CLIP)]) if repair else j_np   # voice from another clip of the same identity
            j, jv = torch.tensor(j_np, device=dev), torch.tensor(jv_np, device=dev)
            xf = (XF[tr[j]] - mf) / sf_; xv = (XV[tr[jv]] - mv) / sv
            if p_in > 0:
                xf = Fn.dropout(xf, p_in, True); xv = Fn.dropout(xv, p_in, True)
            fused, logits, ft, vt = model(xf, xv); lab = ytr[j]
            loss = Fn.cross_entropy(logits, lab) + ALPHA * opl_loss(fused, lab)
            if align == 'mse':
                loss = loss + w_al * ((ft - vt) ** 2).mean()
            elif align == 'nce':
                loss = loss + w_al * nce_loss(ft, vt, lab, tau)
            opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, XF, XV, ch
R4 = ['ecapa', 'wavlm_sv', 'r293', 'redim', 'audeer']; VOICE = R4 + ['whisper']
F_RAW = build_train(['clip']); V_full = build_train(VOICE); U_NAP, n_d = nap_basis(V_full[N_CH:], NAP_K); V_RAW = nap_apply(V_full, U_NAP)
note('harness ready: %d challenge rows, %d MAV rows, face %d-d, voice %d-d after NAP-%d (%d language-difference vectors)' % (N_CH, len(EXT), F_RAW.shape[1], V_RAW.shape[1], NAP_K, n_d))

In [ ]:
# Cell 5 [NB42 v1 6-Nov] - CV: each public cross-modal block in system E's recipe (face CLIP | voice R4 + whisper, NAP-8; CE + 2*OPL + 0.5*L_out tau 0.07).
#          X_<m> = both blocks of model m added; F_<m> = its face block only; V_<m> = its voice block only. R0_ref copied from nb39_cv.csv.
#          Stage A usual folds seeds 1-3; stage B NEW folds + seeds 4-6 (the only stage that counts). Training-side only; per-trial scoring.
SUMMARY = list(NOTES42) + SUMMARY
d = np.load(os.path.join(FEATDIR, 'xmodal.npz'))
XBLOCKS = sorted(set(k.rsplit('_', 1)[0] for k in d.files))
for b in XBLOCKS:
    FEAT[b] = {s: d['%s_%s' % (b, s)] for s in ['train', 'mav', 'extra']}
    assert all(len(FEAT[b][s]) == len(FEAT['clip'][s]) for s in ['train', 'mav', 'extra']), '%s rows do not match - tell Claude' % b
XMODELS = sorted(set(b.split('_')[0] for b in XBLOCKS)); note('cross-modal blocks available: ' + ', '.join(XBLOCKS))
def lout_loss(ft, vt, lab, tau):
    S = Fn.normalize(ft, dim=1) @ Fn.normalize(vt, dim=1).t() / tau; P = (lab[:, None] == lab[None, :]).float()
    def one(M):
        return -(P * Fn.log_softmax(M, 1)).sum(1).div(P.sum(1)).mean()
    return 0.5 * (one(S) + one(S.t()))
def train_e(F_raw, V_raw, all_tr, vsets, seed, w_al=0.5, tau=0.07):
    """system E's recipe (= NB35 train_lout = NB39 train_e): 16 ids x 8 clips, CE + 2*OPL + 0.5*L_out, early stopping on the fold's pairs"""
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev); model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, MAX_EP, None, 0
    for ep in range(1, MAX_EP + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), P_ID, replace=False)
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            fused, logits, ft, vt = model((XF[tr[j]] - mf) / sf_, (XV[tr[j]] - mv) / sv); lab = ytr[j]
            loss = Fn.cross_entropy(logits, lab) + ALPHA * opl_loss(fused, lab) + w_al * lout_loss(ft, vt, lab, tau)
            opt.zero_grad(); loss.backward(); opt.step()
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(best_state); return model, best_ep, XF, XV
COMBOS = {}
for m in XMODELS:
    COMBOS['X_' + m] = (['clip', m + '_face'], VOICE + [m + '_voice'])
for m in XMODELS:
    COMBOS['F_' + m] = (['clip', m + '_face'], VOICE); COMBOS['V_' + m] = (['clip'], VOICE + [m + '_voice'])
def inputs(fn_, vn_):
    F_raw = build_train(fn_); V_full = build_train(vn_); U, _ = nap_basis(V_full[N_CH:], NAP_K); return F_raw, nap_apply(V_full, U)
rngB = np.random.default_rng(7); foldsB = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rngB.shuffle(members)
    for i, c in enumerate(members):
        foldsB[i % K].append(c)
FOLD_B = []
for k in range(K):
    val_ids = set(foldsB[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_B.append((tr_idx, [make_pairs(va_idx, seed=200 + k), make_pairs(va_idx, same_gender_neg=True, seed=300 + k)]))
CSV42 = os.path.join(RESDIR, 'nb42_cv.csv')
if not os.path.exists(CSV42):
    r = pd.read_csv(os.path.join(RESDIR, 'nb39_cv.csv')); r = r[r.combo == 'R0_ref'][['stage', 'combo', 'seed', 'fold', 'best_ep', 'std', 'gender']].copy()
    assert len(r) == 30, 'R0_ref incomplete in nb39_cv.csv - tell Claude'
    r.to_csv(CSV42, index=False); print('R0_ref copied from NB39 (30 runs)')
done42 = pd.read_csv(CSV42); t0 = time.time()
def run42(stage, names, seeds, fold_data):
    global done42
    for name in names:
        todo = [(seed, k) for seed in seeds for k in range(K) if not ((done42.stage == stage) & (done42.combo == name) & (done42.seed == seed) & (done42.fold == k)).any()]
        if not todo:
            continue
        F_raw, V_raw = inputs(*COMBOS[name])
        for seed in seeds:
            for k in range(K):
                if (seed, k) not in todo:
                    continue
                tr_idx, vsets = fold_data[k]
                try:
                    model, best_ep, XF, XV = train_e(F_raw, V_raw, np.concatenate([tr_idx, EXT]), vsets, seed)
                    e_std, e_gen = [eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets]
                except Exception as e:
                    print(name, 'FAILED', repr(e)[:300]); best_ep, e_std, e_gen = -1, 50.0, 50.0
                row = pd.DataFrame([dict(stage=stage, combo=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
                row.to_csv(CSV42, mode='a', header=False, index=False); done42 = pd.concat([done42, row])
            print('%s %-10s seed %d done | %.0f min' % (stage, name, seed, (time.time() - t0) / 60))
        del F_raw, V_raw
def tab42(stage):
    R = pd.read_csv(CSV42); R = R[R.stage == stage].copy(); R['overall'] = (R['std'] + R['gender']) / 2
    return R.groupby(['combo', 'seed'])[['std', 'gender', 'overall', 'best_ep']].mean().groupby('combo').mean().round(2).sort_values('overall')
def adopt(TB, c):
    refB = TB.loc['R0_ref']; dB, dg, ds = TB.loc[c, 'overall'] - refB['overall'], TB.loc[c, 'gender'] - refB['gender'], TB.loc[c, 'std'] - refB['std']
    return (dB <= -0.4 or (dg <= -0.5 and ds <= 0.2)), dB, ds, dg
run42('A', list(COMBOS), [1, 2, 3], FOLD_DATA)
run42('B', list(COMBOS), [4, 5, 6], FOLD_B)
TA, TB = tab42('A'), tab42('B')
note('NB42 STAGE A (usual folds, seeds 1-3; R0_ref = system E recipe):'); note(TA.to_string())
note('NB42 STAGE B (NEW folds + NEW seeds 4-6):'); note(TB.to_string())
note('VERDICTS (stage B only: overall <= -0.4, or gender <= -0.5 with std <= +0.2 vs R0_ref):')
WIN = []
for c in COMBOS:
    ok, dB, ds, dg = adopt(TB, c)
    note('%-8s %s: A %+.2f | B %+.2f (std %+.2f, gender %+.2f) -> %s' % (c, {'X': 'both blocks', 'F': 'face block', 'V': 'voice block'}[c[0]] + ' of ' + c[2:], TA.loc[c, 'overall'] - TA.loc['R0_ref', 'overall'], dB, ds, dg, 'ADOPT' if ok else 'reject'))
    if ok:
        WIN.append(c)
if WIN:
    best_c = min(WIN, key=lambda c: TB.loc[c, 'overall'])
    note('NEXT: ADOPTED (%s); best %s. Claude builds NB42b: the winning block(s) re-extracted on VoxCeleb parts test + 1 (faces re-pulled from the mp4 zips), then the NB20 system.' % (', '.join(WIN), best_c))
else:
    note('NEXT: nothing adopted; the public cross-modal models add nothing to the frozen stack.')
print('=' * 70); print('SUMMARY  (copy everything between the lines and paste to Claude)'); print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)
